In [5]:
import cv2
import numpy as np

def region_of_interest(img):
    """WIDENED: Captures more of the peripheral road surface."""
    height, width = img.shape[:2]
    polygons = np.array([
        [(int(width * 0.05), height),
         (int(width * 0.45), int(height * 0.6)),
         (int(width * 0.55), int(height * 0.6)),
         (int(width * 0.95), height)]
    ], dtype=np.int32)

    mask = np.zeros_like(img)
    cv2.fillPoly(mask, polygons, 255)
    return cv2.bitwise_and(img, mask)

def draw_lines(img, lines):
    """Draws the computed lines onto a blank canvas."""
    line_img = np.zeros_like(img)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            # Bright green lines, 5 pixels thick
            cv2.line(line_img, (x1, y1), (x2, y2), (0, 255, 0), 5)
    return line_img

def process_lane_frame(frame):
    """LOOSENED: Allows for dashed lines and slight gaps."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150)
    masked_edges = region_of_interest(edges)

    lines = cv2.HoughLinesP(masked_edges, 2, np.pi/180, 50,
                            minLineLength=40, maxLineGap=20)

    line_frame = draw_lines(frame, lines)
    return cv2.addWeighted(frame, 0.8, line_frame, 1.0, 0)

def pipeline_executor(input_path, output_path):
    cap = cv2.VideoCapture(input_path)

    if not cap.isOpened():
        print(f"Error: Could not open video file {input_path}")
        return

    # Corrected clean integer lookups
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Grab the raw FPS value cleanly
    raw_fps = cap.get(cv2.CAP_PROP_FPS)
    fps = raw_fps if raw_fps > 0 else 30.0

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    print(f"Processing {input_path}... Extracting lanes.")

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        processed = process_lane_frame(frame)
        out.write(processed)
        frame_count += 1

    cap.release()
    out.release()
    print(f"Finished! Processed {frame_count} frames. Saved tracked video to: {output_path}")

# Run the pipeline on your footage
INPUT_VIDEO = "dashcam2.mp4"   # Make sure this matches your uploaded file name exactly
OUTPUT_VIDEO = "tracked_lanes_output.mp4"

pipeline_executor(INPUT_VIDEO, OUTPUT_VIDEO)

Processing dashcam2.mp4... Extracting lanes.
Finished! Processed 818 frames. Saved tracked video to: tracked_lanes_output.mp4


In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.9 MB/s eta 0:00:00


In [4]:

import cv2
import numpy as np
import os
from ultralytics import YOLO

# ==========================================
# 1. SETUP PATHS & DOWNLOAD SAMPLE VIDEO
# ==========================================
video_dir = "vision_data"
os.makedirs(video_dir, exist_ok=True)

input_video_path = "dashcam1.mp4"
output_video_path = os.path.join(video_dir, "combined_perception.mp4")

if not os.path.exists(input_video_path):
    print("Downloading sample driving video clip...")
    # Downloading a lightweight, public driving clip for testing
    os.system(f"wget -O {input_video_path} https://vocalremover.org/shared/sample.mp4")

# ==========================================
# 2. HELPER FUNCTIONS FOR LANE DETECTION
# ==========================================
def get_roi_mask(image):
    """Isolates the lower triangular region where the road lanes exist."""
    height, width = image.shape[:2]
    # Define a triangular mask targeting the bottom center of the screen
    polygons = np.array([
        [(int(width * 0.1), height), (int(width * 0.5), int(height * 0.6)), (int(width * 0.9), height)]
    ])
    mask = np.zeros_like(image)
    cv2.fillPoly(mask, polygons, 255)
    return cv2.bitwise_and(image, mask)

def draw_lane_lines(image, lines):
    """Draws the mathematical Hough lines onto the image frame."""
    lane_image = np.zeros_like(image)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            # Draw green lane lines
            cv2.line(lane_image, (x1, y1), (x2, y2), (0, 255, 0), 5)
    return cv2.addWeighted(image, 0.8, lane_image, 1.0, 0)

# ==========================================
# 3. INITIALIZE STREAMS & NEURAL NETWORK
# ==========================================
# Automatically downloads the 6MB pre-trained weights file instantly
yolo_model = YOLO("yolov8n.pt")

cap = cv2.VideoCapture(input_video_path)
if not cap.isOpened():
    raise FileNotFoundError("Could not open input video stream.")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Initialize the video writer to record the combined frames
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

print(f"Processing Stream: {frame_width}x{frame_height} @ {fps} FPS. Total frames: {total_frames}")

# ==========================================
# 4. SEQUENTIAL PROCESSING LOOP
# ==========================================
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # --- STREAM A: CLASSICAL LANE DETECTION ---
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    canny = cv2.Canny(blur, 50, 150)
    roi = get_roi_mask(canny)

    # Mathematical Hough Line Transform to detect road markers
    lines = cv2.HoughLinesP(roi, 2, np.pi/180, 100, minLineLength=40, maxLineGap=5)
    processed_frame = draw_lane_lines(frame, lines)

    # --- STREAM B: DEEP LEARNING OBJECT DETECTION ---
    # Run YOLOv8 on the GPU (device=0); filter for common traffic classes
    # 0: person, 2: car, 3: motorcycle, 5: bus, 7: truck, 16: dog, 19: cow
    yolo_results = yolo_model(processed_frame, device=0, classes=[0, 2, 3, 5, 7, 16, 19], verbose=False)[0]

    # Extract boxes and draw them onto our frame
    for box in yolo_results.boxes:
        # Bounding box coordinates
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])
        label = f"{yolo_model.names[cls_id]} {conf:.2f}"

        # Draw bounding box (Blue) and label text
        cv2.rectangle(processed_frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(processed_frame, label, (x1, max(y1 - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    # --- WRITE COMBINED FRAME TO DISK ---
    out.write(processed_frame)

    frame_count += 1
    if frame_count % 50 == 0:
        print(f"Progress: Combined perception applied to frame {frame_count}/{total_frames}")

# Clean up memory buffers safely
cap.release()
out.release()

print("\n==============================================================")
print(f"SUCCESS: Dual Perception Video generated safely!")
print(f"Output saved to: {output_video_path}")
print("Open the left sidebar panel in Colab to download the final MP4.")
print("==============================================================")

Processing Stream: 1920x1080 @ 29.97002997002997 FPS. Total frames: 1252
Progress: Combined perception applied to frame 50/1252
Progress: Combined perception applied to frame 100/1252
Progress: Combined perception applied to frame 150/1252
Progress: Combined perception applied to frame 200/1252
Progress: Combined perception applied to frame 250/1252
Progress: Combined perception applied to frame 300/1252
Progress: Combined perception applied to frame 350/1252
Progress: Combined perception applied to frame 400/1252
Progress: Combined perception applied to frame 450/1252
Progress: Combined perception applied to frame 500/1252
Progress: Combined perception applied to frame 550/1252
Progress: Combined perception applied to frame 600/1252
Progress: Combined perception applied to frame 650/1252
Progress: Combined perception applied to frame 700/1252
Progress: Combined perception applied to frame 750/1252
Progress: Combined perception applied to frame 800/1252
Progress: Combined perception ap

In [7]:

import cv2
import numpy as np
import os
from ultralytics import YOLO

# ==========================================
# 1. SETUP PATHS
# ==========================================
video_dir = "vision_data"
os.makedirs(video_dir, exist_ok=True)
input_video_path = "dashcam1.mp4"
output_video_path = os.path.join(video_dir, "dynamic_aligned_perception.mp4")

if not os.path.exists(input_video_path):
    os.system(f"wget -O {input_video_path} https://vocalremover.org/shared/sample.mp4")

# ==========================================
# 2. CALIBRATION CONSTANTS
# ==========================================
FOCAL_LENGTH_PX = 500.0
REAL_HEIGHTS_METERS = {0: 1.70, 2: 1.50, 3: 1.00, 5: 2.80, 7: 3.00, 16: 0.60, 19: 1.50}

# ==========================================
# 3. DYNAMIC LANE MATH & PROJECTION
# ==========================================
def average_lane_lines(image, lines):
    """Averages Hough lines into a single solid left and right lane boundary."""
    left_fit, right_fit = [], []
    height = image.shape[0]

    if lines is None:
        return None

    for line in lines:
        x1, y1, x2, y2 = line[0]
        if x1 == x2: continue # Prevent division by zero
        slope = (y2 - y1) / (x2 - x1)
        intercept = y1 - slope * x1

        # Separate lines based on slope direction
        if slope < -0.3: # Left lane
            left_fit.append((slope, intercept))
        elif slope > 0.3: # Right lane
            right_fit.append((slope, intercept))

    # Calculate average slopes and intercepts
    lines_to_draw = []
    y1 = height # Bottom of the frame
    y2 = int(height * 0.6) # Horizon line

    if left_fit:
        left_avg = np.average(left_fit, axis=0)
        x1_l = int((y1 - left_avg[1]) / left_avg[0])
        x2_l = int((y2 - left_avg[1]) / left_avg[0])
        lines_to_draw.append(((x1_l, y1, x2_l, y2), left_avg))

    if right_fit:
        right_avg = np.average(right_fit, axis=0)
        x1_r = int((y1 - right_avg[1]) / right_avg[0])
        x2_r = int((y2 - right_avg[1]) / right_avg[0])
        lines_to_draw.append(((x1_r, y1, x2_r, y2), right_avg))

    return lines_to_draw

def draw_dynamic_grid(image, averaged_lines):
    """Draws distance markers strictly between the dynamically detected lane boundaries."""
    if not averaged_lines or len(averaged_lines) != 2:
        return image # Need both lines to draw a connecting grid

    overlay = image.copy()
    height = image.shape[0]

    # Extract slopes and intercepts for Left and Right lines
    left_eq, right_eq = averaged_lines[0][1], averaged_lines[1][1]

    # Define Y-levels for our distance intervals
    grid_levels = [
        (int(height * 0.95), "1.0m", (0, 0, 255)),   # Red
        (int(height * 0.85), "3.0m", (0, 255, 255)), # Yellow
        (int(height * 0.75), "5.0m", (0, 255, 0))    # Green
    ]

    for y_pos, label, color in grid_levels:
        # Mathematically calculate where the left and right lines intersect this Y level
        left_x = int((y_pos - left_eq[1]) / left_eq[0])
        right_x = int((y_pos - right_eq[1]) / right_eq[0])

        # Draw horizontal connecting bar
        cv2.line(overlay, (left_x, y_pos), (right_x, y_pos), color, 3)
        cv2.putText(overlay, label, (left_x - 45, y_pos + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)

    # Draw the solid outer lane boundaries
    for line_coords, _ in averaged_lines:
        x1, y1, x2, y2 = line_coords
        cv2.line(overlay, (x1, y1), (x2, y2), (255, 255, 255), 3)

    return cv2.addWeighted(overlay, 0.5, image, 0.5, 0)

def get_roi_mask(image):
    height, width = image.shape[:2]
    polygons = np.array([[(int(width * 0.1), height), (int(width * 0.5), int(height * 0.55)), (int(width * 0.9), height)]])
    mask = np.zeros_like(image)
    cv2.fillPoly(mask, polygons, 255)
    return cv2.bitwise_and(image, mask)

# ==========================================
# 4. INITIALIZE UPGRADED AI MODEL
# ==========================================
# UPGRADE: Switching from yolov8n.pt to yolov8m.pt (Medium) for better accuracy
yolo_model = YOLO("yolov8m.pt")

cap = cv2.VideoCapture(input_video_path)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# ==========================================
# 5. EXECUTION LOOP
# ==========================================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # --- CLASSICAL LANE DETECTOR ---
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    canny = cv2.Canny(blur, 50, 150)
    roi = get_roi_mask(canny)

    # Extract chaotic lines, then average them into solid boundaries
    raw_lines = cv2.HoughLinesP(roi, 2, np.pi/180, 100, minLineLength=40, maxLineGap=5)
    averaged_lines = average_lane_lines(frame, raw_lines)

    # Draw the dynamic perspective grid anchored to those specific boundaries
    processed_frame = draw_dynamic_grid(frame, averaged_lines)

    # --- DEEP LEARNING (UPGRADED) ---
    tracked_classes = list(REAL_HEIGHTS_METERS.keys())
    # UPGRADE: Added strict confidence threshold (conf=0.45) to prevent misclassifications
    yolo_results = yolo_model(processed_frame, device=0, classes=tracked_classes, conf=0.45, verbose=False)[0]

    for box in yolo_results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        bbox_height_px = y2 - y1
        if bbox_height_px <= 0: continue

        cls_id = int(box.cls[0])
        distance_m = (REAL_HEIGHTS_METERS.get(cls_id, 1.5) * FOCAL_LENGTH_PX) / bbox_height_px

        display_label = f"{yolo_model.names[cls_id]} [{distance_m:.1f}m]"
        box_color = (0, 0, 255) if distance_m < 2.0 else (0, 255, 255) if distance_m < 5.0 else (255, 0, 0)

        cv2.rectangle(processed_frame, (x1, y1), (x2, y2), box_color, 2)
        cv2.putText(processed_frame, display_label, (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)

    out.write(processed_frame)

cap.release()
out.release()
print(f"SUCCESS: Download 'dynamic_aligned_perception.mp4' from your Colab files tab.")

SUCCESS: Download 'dynamic_aligned_perception.mp4' from your Colab files tab.


In [9]:
from google.colab import files

print("Initiating download for the pre-trained YOLOv8 Medium model...")
files.download("yolov8m.pt")

Initiating download for the pre-trained YOLOv8 Medium model...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import cv2
import numpy as np
import os
import torch
from ultralytics import YOLO

# ==============================================================================
# 1. HARDWARE ACCELERATION SETUP
# ==============================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("--- COGNITIVE PERCEPTION STACK INITIALIZING ---")
print(f"Primary hardware accelerator routed to: [{device.upper()}]")

# ==============================================================================
# 2. LOCAL FILE PATHS
# ==============================================================================
# Place your driving video file in the same directory as this script
INPUT_VIDEO = "dashcam.mp4"
OUTPUT_VIDEO = "predicted_output.mp4"  # The final generated video file
MODEL_WEIGHTS = "yolov8m.pt"

if not os.path.exists(MODEL_WEIGHTS):
    raise FileNotFoundError(f"Could not find '{MODEL_WEIGHTS}' in the local directory. Please verify the download.")
if not os.path.exists(INPUT_VIDEO):
    raise FileNotFoundError(f"Missing input video stream. Please drop '{INPUT_VIDEO}' into this folder.")

# ==============================================================================
# 3. MATHEMATICAL ALIGNMENT ALGORITHMS
# ==============================================================================
def average_lane_lines(image, lines):
    left_fit, right_fit = [], []
    height = image.shape[0]
    if lines is None:
        return None

    for line in lines:
        x1, y1, x2, y2 = line[0]
        if x1 == x2:
            continue
        slope = (y2 - y1) / (x2 - x1)
        intercept = y1 - slope * x1

        if slope < -0.3:    # Left lane
            left_fit.append((slope, intercept))
        elif slope > 0.3:   # Right lane
            right_fit.append((slope, intercept))

    lines_to_draw = []
    y1 = height
    y2 = int(height * 0.6)

    if left_fit:
        left_avg = np.average(left_fit, axis=0)
        x1_l = int((y1 - left_avg[1]) / left_avg[0])
        x2_l = int((y2 - left_avg[1]) / left_avg[0])
        lines_to_draw.append(((x1_l, y1, x2_l, y2), left_avg))

    if right_fit:
        right_avg = np.average(right_fit, axis=0)
        x1_r = int((y1 - right_avg[1]) / right_avg[0])
        x2_r = int((y2 - right_avg[1]) / right_avg[0])
        lines_to_draw.append(((x1_r, y1, x2_r, y2), right_avg))

    return lines_to_draw

def draw_dynamic_grid(image, averaged_lines):
    """
    Draws the perspective projection lines. If dynamic lines are lost,
    it falls back to a stationary reverse camera template so the grid never disappears.
    """
    overlay = image.copy()
    height, width = image.shape[:2]

    # Trackers found: Draw dynamically aligned perspective lanes
    if averaged_lines and len(averaged_lines) == 2:
        left_eq, right_eq = averaged_lines[0][1], averaged_lines[1][1]

        grid_levels = [
            (int(height * 0.95), "1.0m", (0, 0, 255)),   # Red Zone
            (int(height * 0.85), "3.0m", (0, 255, 255)), # Yellow Zone
            (int(height * 0.75), "5.0m", (0, 255, 0))    # Green Zone
        ]

        for y_pos, label, color in grid_levels:
            left_x = int((y_pos - left_eq[1]) / left_eq[0])
            right_x = int((y_pos - right_eq[1]) / right_eq[0])
            cv2.line(overlay, (left_x, y_pos), (right_x, y_pos), color, 3)
            cv2.putText(overlay, label, (left_x - 45, y_pos + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)

        for line_coords, _ in averaged_lines:
            x1, y1, x2, y2 = line_coords
            cv2.line(overlay, (x1, y1), (x2, y2), (255, 255, 255), 3)

    # Trackers lost: Draw stationary backup camera guideline matrix safely
    else:
        distance_zones = [
            (int(width * 0.15), int(width * 0.85), int(height * 0.95), "1.0m", (0, 0, 255)),
            (int(width * 0.25), int(width * 0.75), int(height * 0.80), "3.0m", (0, 255, 255)),
            (int(width * 0.32), int(width * 0.68), int(height * 0.70), "5.0m", (0, 255, 0))
        ]
        # Draw side track rails
        cv2.line(overlay, (distance_zones[2][0], distance_zones[2][2]), (distance_zones[0][0], distance_zones[0][2]), (255, 255, 255), 2)
        cv2.line(overlay, (distance_zones[2][1], distance_zones[2][2]), (distance_zones[0][1], distance_zones[0][2]), (255, 255, 255), 2)

        # Draw cross bars
        for left_x, right_x, y_pos, label, color in distance_zones:
            cv2.line(overlay, (left_x, y_pos), (right_x, y_pos), color, 3)
            cv2.putText(overlay, label, (left_x - 45, y_pos + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)

    return cv2.addWeighted(overlay, 0.5, image, 0.5, 0)

def get_roi_mask(image):
    height, width = image.shape[:2]
    polygons = np.array([[(int(width * 0.1), height), (int(width * 0.5), int(height * 0.55)), (int(width * 0.9), height)]])
    mask = np.zeros_like(image)
    cv2.fillPoly(mask, polygons, 255)
    return cv2.bitwise_and(image, mask)

# ==============================================================================
# 4. INITIALIZATION ENGINE & VIDEO WRITER
# ==============================================================================
yolo_model = YOLO(MODEL_WEIGHTS)
cap = cv2.VideoCapture(INPUT_VIDEO)

# Extract original video properties
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Initialize the video encoder to save the file locally
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (frame_width, frame_height))

FOCAL_LENGTH_PX = 500.0
REAL_HEIGHTS_METERS = {0: 1.70, 2: 1.50, 3: 1.00, 5: 2.80, 7: 3.00, 16: 0.60, 19: 1.50}

print(f"\n--- Local Headless Processing Started ---")
print(f"Encoding properties: {frame_width}x{frame_height} at {fps} FPS.")
print(f"Processing frames... Saving output directly to: {OUTPUT_VIDEO}")

# ==============================================================================
# 5. STREAM PIPELINE EXECUTION LOOP (HEADLESS)
# ==============================================================================
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # --- PHASE A: LANE DETECTION & PROJECTION GRID ---
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    canny = cv2.Canny(blur, 50, 150)
    roi = get_roi_mask(canny)

    raw_lines = cv2.HoughLinesP(roi, 2, np.pi/180, 100, minLineLength=40, maxLineGap=5)
    averaged_lines = average_lane_lines(frame, raw_lines)
    processed_frame = draw_dynamic_grid(frame, averaged_lines)

    # --- PHASE B: OBJECT TRACKING & DEPTH MAPPING ---
    tracked_classes = list(REAL_HEIGHTS_METERS.keys())
    yolo_results = yolo_model(processed_frame, device=device, classes=tracked_classes, conf=0.45, verbose=False)[0]

    for box in yolo_results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        bbox_height_px = y2 - y1
        if bbox_height_px <= 0:
            continue

        cls_id = int(box.cls[0])
        distance_m = (REAL_HEIGHTS_METERS.get(cls_id, 1.5) * FOCAL_LENGTH_PX) / bbox_height_px

        display_label = f"{yolo_model.names[cls_id]} [{distance_m:.1f}m]"
        box_color = (0, 0, 255) if distance_m < 2.0 else (0, 255, 255) if distance_m < 5.0 else (255, 0, 0)

        cv2.rectangle(processed_frame, (x1, y1), (x2, y2), box_color, 2)
        cv2.putText(processed_frame, display_label, (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)

    # --- PHASE C: WRITE GENERATED FRAME TO HARD DRIVE ---
    out.write(processed_frame)

    frame_count += 1
    if frame_count % 50 == 0:
        print(f"Progress status: Processed and saved {frame_count}/{total_frames} frames...")

# Safely close files and flush hardware handles
cap.release()
out.release()

print("\n==============================================================")
print(f"SUCCESS: Predicted video generation complete!")
print(f"The final annotated file has been saved as: {OUTPUT_VIDEO}")
print("==============================================================")

--- COGNITIVE PERCEPTION STACK INITIALIZING ---
Primary hardware accelerator routed to: [CUDA]

--- Local Headless Processing Started ---
Encoding properties: 1280x720 at 29.97002997002997 FPS.
Processing frames... Saving output directly to: predicted_output.mp4
Progress status: Processed and saved 50/7614 frames...
Progress status: Processed and saved 100/7614 frames...
Progress status: Processed and saved 150/7614 frames...
Progress status: Processed and saved 200/7614 frames...
Progress status: Processed and saved 250/7614 frames...
Progress status: Processed and saved 300/7614 frames...
Progress status: Processed and saved 350/7614 frames...
Progress status: Processed and saved 400/7614 frames...
Progress status: Processed and saved 450/7614 frames...
Progress status: Processed and saved 500/7614 frames...
Progress status: Processed and saved 550/7614 frames...
Progress status: Processed and saved 600/7614 frames...
Progress status: Processed and saved 650/7614 frames...
Progress s